# Tech Challenge | Olist | Análise Executiva

Notebook principal do projeto.

**Objetivo:** transformar os dados transacionais do Olist em insights executivos sobre crescimento, logística e satisfação do cliente, com recomendações acionáveis para investidores e acionistas.


## 1. Importação e leitura dos dados

In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing

ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"
CHARTS = OUTPUTS / "charts"


In [ ]:

orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")


## 2. Limpeza e transformação

In [ ]:

for col in [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

for col in ["review_creation_date", "review_answer_timestamp"]:
    reviews[col] = pd.to_datetime(reviews[col], errors="coerce")

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")


In [ ]:

payments_agg = payments.groupby("order_id", as_index=False).agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type_primary=("payment_type", lambda s: s.mode().iat[0] if not s.mode().empty else s.iloc[0]),
    payment_records=("payment_type", "size"),
)

reviews_agg = reviews.groupby("order_id", as_index=False).agg(
    review_score=("review_score", "mean"),
    has_review_comment=("review_comment_message", lambda s: int(s.notna().any())),
)

item_agg = items.groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    unique_sellers=("seller_id", "nunique"),
)


## 3. Joins principais com pandas

In [ ]:

orders_enriched = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
    .merge(item_agg, on="order_id", how="left")
)

orders_enriched["purchase_month"] = orders_enriched["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
orders_enriched["delivery_days"] = (
    orders_enriched["order_delivered_customer_date"] - orders_enriched["order_purchase_timestamp"]
).dt.days
orders_enriched["estimated_gap_days"] = (
    orders_enriched["order_delivered_customer_date"] - orders_enriched["order_estimated_delivery_date"]
).dt.days
orders_enriched["is_late"] = orders_enriched["estimated_gap_days"] > 0

orders_enriched.head()


## 4. Base analítica por item

In [ ]:

items_enriched = (
    items
    .merge(orders[[
        "order_id", "customer_id", "order_status", "order_purchase_timestamp",
        "order_delivered_customer_date", "order_estimated_delivery_date"
    ]], on="order_id", how="left")
    .merge(customers[["customer_id", "customer_unique_id", "customer_state", "customer_city"]], on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(translation, on="product_category_name", how="left")
    .merge(sellers[["seller_id", "seller_state", "seller_city"]], on="seller_id", how="left")
    .merge(reviews_agg[["order_id", "review_score"]], on="order_id", how="left")
)

items_enriched["category_en"] = (
    items_enriched["product_category_name_english"]
    .fillna(items_enriched["product_category_name"])
    .fillna("unknown")
)

items_enriched.head()


## 5. KPIs principais

In [ ]:

delivered = orders_enriched.query("order_status == 'delivered'").copy()

kpis = {
    "Pedidos entregues": int(delivered["order_id"].nunique()),
    "GMV entregue": round(float(delivered["payment_value"].sum()), 2),
    "Ticket médio": round(float(delivered["payment_value"].mean()), 2),
    "Review médio": round(float(delivered["review_score"].mean()), 2),
    "Prazo médio (dias)": round(float(delivered["delivery_days"].mean()), 2),
    "Taxa de atraso": round(float(delivered["is_late"].mean()), 4),
}
kpis


## 6. Visões executivas

In [ ]:

monthly = delivered.groupby("purchase_month").agg(
    orders=("order_id", "nunique"),
    revenue=("payment_value", "sum"),
    avg_ticket=("payment_value", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
    avg_review=("review_score", "mean"),
    late_rate=("is_late", "mean"),
).reset_index()

state = delivered.groupby("customer_state").agg(
    orders=("order_id", "nunique"),
    revenue=("payment_value", "sum"),
    avg_ticket=("payment_value", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
    late_rate=("is_late", "mean"),
    avg_review=("review_score", "mean"),
).sort_values("revenue", ascending=False).reset_index()

category = items_enriched.query("order_status == 'delivered'").groupby("category_en").agg(
    revenue=("price", "sum"),
    orders=("order_id", "nunique"),
    avg_review=("review_score", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
).sort_values("revenue", ascending=False).reset_index()

monthly.head(), state.head(), category.head()


## 7. Gráfico de receita mensal

In [ ]:

plot_df = monthly[(monthly["purchase_month"] >= "2017-01-01") & (monthly["purchase_month"] < "2018-08-01")].copy()

plt.figure(figsize=(12, 5))
plt.plot(plot_df["purchase_month"], plot_df["revenue"])
plt.title("Receita mensal - pedidos entregues")
plt.grid(True, alpha=0.3)
plt.show()


## 8. Impacto do atraso no review

In [ ]:

late_comp = delivered.groupby("is_late")["review_score"].mean().reset_index()
late_comp["status"] = late_comp["is_late"].map({False: "No prazo", True: "Atrasado"})
late_comp


## 9. Forecast simples (cenário tendencial)

In [ ]:

model = ExponentialSmoothing(plot_df.set_index("purchase_month")["revenue"], trend="add", seasonal=None).fit(optimized=True)
forecast = model.forecast(3)
forecast


## 10. Principais conclusões

- O negócio acelerou fortemente entre 2017 e 2018.
- A logística é o maior driver de satisfação.
- Atraso reduz drasticamente a nota média do cliente.
- Há espaço para ganho operacional em estados com alta receita e atraso elevado.
- Categorias grandes com review abaixo de 4,0 devem virar prioridade de ação.
